In [13]:
# some imports 
import numpy as np 
import networkx as nx
import copy 
from itertools import product
from collections import Counter
import matplotlib.pyplot as plt
from numpy import random
#D-Wave Ocean SDK related
import dimod 
from dwave.system import DWaveSampler
from dwave.samplers import SimulatedAnnealingSampler
# from dwave.cloud import Client
# client = Client.from_config()
from dwave.embedding import embed_ising, unembed_sampleset
from minorminer import find_embedding
import pickle
from pathlib import Path

seed = 1224
random.seed(seed)



In [ ]:
embeddings = pickle.loads(Path("../pkl/embeddings.pkl").read_bytes())
SA_sample_sets = pickle.loads(Path("../pkl/SA_sample_sets.pkl").read_bytes())
QA_sample_sets = pickle.loads(Path("../pkl/QA_sample_sets.pkl").read_bytes())
sa_results = pickle.loads(Path("../pkl/sa_results.pkl").read_bytes())
qa_results = pickle.loads(Path("../pkl/qa_results.pkl").read_bytes())
graphs = pickle.loads(Path("../pkl/graphs.pkl").read_bytes())
labels = pickle.loads(Path("../pkl/labels.pkl").read_bytes())

In [15]:
from dwave.samplers import SteepestDescentSolver

SD_solver = SteepestDescentSolver()

def run_SD(graph, SA_sampleset, label):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")

    SD_sampleset = SD_solver.sample_ising(
        h, J,
        initial_states=SA_sampleset,
        num_reads=len(SA_sampleset),
    )

    SA_energies = SA_sampleset.data_vectors["energy"]
    SD_energies = SD_sampleset.data_vectors["energy"]
    return SD_sampleset, {
        "label": label,
        "SA_expected": np.mean(SA_energies),
        "SA_best": SA_energies.min(),
        "SD_expected": np.mean(SD_energies),
        "SD_best": SD_energies.min(),
    }

SD_sample_sets = []
SD_summaries = []
for graph, SA_ss, label in zip(graphs, SA_sample_sets, labels):
    ss, summary = run_SD(graph, SA_ss, label)
    SD_sample_sets.append(ss)
    SD_summaries.append(summary)

In [18]:
print(len(graphs), len(labels), len(SA_sample_sets), len(QA_sample_sets), len(embeddings))

18 18 18 18 18


In [17]:
import pickle
from pathlib import Path
import numpy as np, pandas as pd, networkx as nx, dimod
from dwave.samplers import SteepestDescentSolver

SD = SteepestDescentSolver()

graphs         = pickle.loads(Path("graphs.pkl").read_bytes())
labels         = pickle.loads(Path("labels.pkl").read_bytes())
SA_sample_sets = pickle.loads(Path("SA_sample_sets.pkl").read_bytes())
QA_sample_sets = pickle.loads(Path("QA_sample_sets.pkl").read_bytes())

assert len(graphs) == len(labels) == len(SA_sample_sets) == len(QA_sample_sets)

def logical_bqm(graph):
    h = nx.get_node_attributes(graph, "weight")
    J = nx.get_edge_attributes(graph, "weight")
    h = {n: h.get(n, 0.0) for n in graph.nodes}
    return dimod.BinaryQuadraticModel.from_ising(h, J)

def weighted_stats(e, w):
    w = np.asarray(w, dtype=float)
    mean = np.average(e, weights=w)
    return {"best": e.min(), "expected": mean,
            "std": np.sqrt(np.average((e - mean) ** 2, weights=w)),
            "frac_at_best": w[np.isclose(e, e.min())].sum() / w.sum(),
            "distinct": len(e)}

def evaluate(graph, label, SA_ss, QA_ss):
    bqm = logical_bqm(graph)
    rows = []
    for name, ss in (("SA", SA_ss), ("QA", QA_ss)):
        assert set(ss.variables) == set(bqm.variables), f"{label} {name}: variable mismatch"
        occ = ss.record.num_occurrences
        e_raw = bqm.energies(ss)                      # logical objective, not the stored energies

        # one row per read so occurrence counts are preserved through postprocessing
        init = np.repeat(ss.record.sample, occ, axis=0)
        pp = SD.sample(bqm, initial_states=(init, list(ss.variables)))
        e_before = np.repeat(e_raw, occ)
        e_after = pp.record.energy
        assert np.all(e_after <= e_before + 1e-9), "SD output not aligned with initial states"
        flips = (pp.record.sample != init).sum(axis=1)

        extra = {"n_reads": int(occ.sum()),
                 "frac_already_local_min": np.mean(flips == 0),
                 "mean_flips": flips.mean(),
                 "chain_break_frac": (ss.record.chain_break_fraction.mean()
                                      if "chain_break_fraction" in ss.record.dtype.names else np.nan)}
        base = {"label": label, "method": name}
        rows.append({**base, "post": False, **weighted_stats(e_raw, occ), **extra})
        rows.append({**base, "post": True,  **weighted_stats(e_after, np.ones_like(e_after)), **extra})
    return rows

rows = []
for g, lab, sa, qa in zip(graphs, labels, SA_sample_sets, QA_sample_sets):
    rows += evaluate(g, lab, sa, qa)

df = pd.DataFrame(rows)
df["variant"] = df["method"] + np.where(df["post"], "+SD", "")
pd.set_option("display.width", 220, "display.max_columns", 30, "display.float_format", "{:.2f}".format)

# 1. Four-way comparison per graph
for stat in ("best", "expected"):
    print(f"\n=== {stat} energy ===")
    print(df.pivot_table(index="label", columns="variant", values=stat)[["SA", "SA+SD", "QA", "QA+SD"]])

# 2. Gain from postprocessing (positive = lower energy)
g = df.pivot_table(index=["label", "method"], columns="post", values=["best", "expected"])
gain = pd.DataFrame({"best_gain": g[("best", False)] - g[("best", True)],
                     "expected_gain": g[("expected", False)] - g[("expected", True)]})
print("\n=== gain from steepest descent ===")
print(gain)

# 3. Sample-property diagnostics (before postprocessing)
print("\n=== diagnostics ===")
print(df[~df["post"]][["label", "method", "n_reads", "distinct", "std", "frac_at_best",
                       "frac_already_local_min", "mean_flips", "chain_break_frac"]])


=== best energy ===
variant                           SA    SA+SD       QA    QA+SD
label                                                          
Binary Barabasi-Albert      -1604.00 -1604.00 -1546.00 -1566.00
Binary Barabasi-Albert (1)  -4159.00 -4159.00 -4087.00 -4141.00
Binary Erdos-Renyi          -1235.00 -1235.00 -1071.00 -1229.00
Binary Full                 -1666.00 -1666.00 -1502.00 -1666.00
Binary Regular              -1344.00 -1344.00 -1320.00 -1328.00
Binary Square               -3713.00 -3713.00 -3695.00 -3703.00
Normal Barabasi-Albert      -1519.39 -1519.39 -1338.87 -1470.54
Normal Barabasi-Albert (1)  -4130.26 -4130.26 -3787.87 -4093.98
Normal Erdos-Renyi          -1205.84 -1205.84 -1072.18 -1203.81
Normal Full                 -1684.68 -1684.68 -1485.08 -1675.56
Normal Regular              -1311.06 -1311.06 -1200.39 -1278.71
Normal Square               -3525.03 -3525.03 -3285.13 -3449.69
Uniform Barabasi-Albert      -902.19  -902.19  -858.48  -885.59
Uniform Barabasi-Al